In [ ]:
# Locate data directory and read in data files
import os
from pathlib import Path
import pandas as pd
from helper_functions import *

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    final_crosswalk_df = pd.read_csv(data_dir / 'final_crosswalk.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()

In [ ]:
final_crosswalk_df[final_crosswalk_df["cik"].duplicated(keep = False)]


In [ ]:
final_crosswalk_df[final_crosswalk_df["standardized_names"].isna()]

In [ ]:
fuzzy_rows = final_crosswalk_df[
    (final_crosswalk_df['fuzzy_matching_score'].str.len() > 0) & 
    (final_crosswalk_df['fuzzy_matching_score'].astype(str) != '[]')
]
fuzzy_rows

# Final crosswalk cleanup for package use

In [ ]:
final_crosswalk_df

In [ ]:
final_crosswalk_df.drop(columns = ['standardized_names', 'matching_type', 'fuzzy_matching_score', 'ineligible_name_matching'])

EDA for tier 1 matches from the python regextable script

In [ ]:
try:
    tier1_matches = pd.read_csv(data_dir / 'matches_stage1_CIK.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()

In [ ]:
tier1_matches['unique_id'] = tier1_matches['unique_id'].astype(str).str.replace('CIK-', '', regex=False)
tier1_matches['unique_id'] = tier1_matches['unique_id'].astype(int)

In [ ]:
overlap_df = final_crosswalk_df.merge(tier1_matches, left_on='cik', right_on='unique_id', how='inner')

In [ ]:
# Use a right merge to keep everything in df2
combined = final_crosswalk_df.merge(tier1_matches, left_on='cik', right_on='unique_id', how='right', indicator=True)

# Filter for 'right_only' to find what's missing from the left
only_in_right = combined[combined['_merge'] == 'right_only']